# D128 — SMTP Demonstration (Do Not Run Automatically)

> **Important:** This notebook contains real email-sending code, but sending is disabled by default. Read each cell before running it.

This lesson sends one test message to a disposable inbox. [Temp Mail](https://temp-mail.org/) supplies the temporary **receiving address**. It does not supply the outgoing SMTP account used in this example.

## How the demonstration works

```text
Python code → Gmail SMTP server → temporary Temp Mail inbox
```

You need:

1. A temporary recipient address copied from Temp Mail.
2. A Gmail address for the sender.
3. A Gmail App Password for SMTP authentication. Do not use or expose your normal account password.

Gmail documents `smtp.gmail.com` with SSL on port 465 or TLS on port 587. This notebook uses SSL on port 465.

## 1. Copy a temporary recipient address

1. Open [https://temp-mail.org/](https://temp-mail.org/).
2. Copy the temporary email address displayed on the page.
3. Paste it into `TEMP_MAIL_RECIPIENT` below.

Temporary inboxes may expire. Use them only for harmless demonstrations, never for private or important information.

In [ ]:
# Copy and paste the address generated by temp-mail.org here.
TEMP_MAIL_RECIPIENT = "paste-temp-mail-address-here"

# This Gmail account would send the demonstration message.
SENDER_EMAIL = "your-sender-address@gmail.com"

# Safety switch: leave False while reading or smoke-testing the notebook.
SEND_REAL_EMAIL = False

## 2. Build the email message

`EmailMessage` creates the sender, recipient, subject, and body in a standard email format. Building the message does not use the network.

In [ ]:
from email.message import EmailMessage


def build_message(sender, recipient):
    message = EmailMessage()
    message["From"] = sender
    message["To"] = recipient
    message["Subject"] = "Python SMTP demonstration"
    message.set_content(
        "Hello! This is a harmless SMTP test from a Python lesson."
    )
    return message


email_message = build_message(SENDER_EMAIL, TEMP_MAIL_RECIPIENT)
print(email_message)

## 3. Define the SMTP function

Defining this function does not send anything. `SMTP_SSL` creates an encrypted connection, `login` authenticates the sender, and `send_message` submits the email.

In [ ]:
import smtplib
import ssl


def send_with_gmail(message, sender_email, app_password):
    ssl_context = ssl.create_default_context()

    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465,
        context=ssl_context,
        timeout=15,
    ) as smtp:
        smtp.login(sender_email, app_password)
        smtp.send_message(message)

## 4. Deliberately guarded send cell

The following cell does **not** send while `SEND_REAL_EMAIL` is `False`. To perform the demonstration intentionally:

1. Confirm the sender and recipient variables.
2. Obtain an App Password from the sender account if it is supported and enabled.
3. Change `SEND_REAL_EMAIL` to `True`.
4. Run the cell and enter the App Password at the hidden prompt.
5. Refresh the Temp Mail inbox.
6. Immediately change the switch back to `False`.

The password is requested with `getpass`, so it is not written into the notebook or displayed as output.

In [ ]:
from getpass import getpass


if SEND_REAL_EMAIL:
    if "@" not in TEMP_MAIL_RECIPIENT:
        raise ValueError("Paste a valid Temp Mail recipient first")
    if not SENDER_EMAIL.endswith("@gmail.com"):
        raise ValueError("Enter the Gmail sender address first")

    gmail_app_password = getpass("Gmail App Password: ")
    send_with_gmail(email_message, SENDER_EMAIL, gmail_app_password)
    print("Message submitted. Check the temporary inbox.")
else:
    print("SAFE MODE: no email was sent")

## Common problems

- **Authentication error:** use an App Password where supported, not the regular Gmail password. Account or administrator settings may restrict SMTP access.
- **Message does not appear:** refresh the temporary inbox and check that its address has not expired. Delivery and spam filtering are not guaranteed.
- **Connection error:** a school, office, VPN, or firewall may block outbound SMTP ports.
- **Different sender provider:** replace the server, port, encryption method, and authentication details with that provider's official settings.

Never commit passwords or paste them directly into notebook cells.

## Summary

- Temp Mail supplies the disposable recipient inbox.
- An SMTP provider supplies the authenticated sending service.
- `EmailMessage` builds the message and `smtplib` submits it.
- The safety switch prevents accidental sending during normal notebook execution.
- Secrets should be entered securely and must not be stored in the notebook.